# 03 — Selección y filtrado

Seleccionar subconjuntos de filas y columnas es la operación más frecuente en análisis de datos. Pandas ofrece varias formas de hacerlo y cada una tiene su caso de uso.

## Setup

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

def find_project_root():
    for parent in [Path.cwd()] + list(Path.cwd().parents):
        if (parent / 'data').exists():
            return parent

ROOT   = find_project_root()
TRAIN  = ROOT / 'data' / 'external' / 'train.csv'
AIR    = ROOT / 'data' / 'external' / 'Listings.csv'
df = pd.read_csv(TRAIN, low_memory=False)
air = pd.read_csv(AIR, encoding='latin-1', low_memory=False)


## Selección de columnas

In [2]:
# Una columna → Series
ventas = df['Sales']
print(type(ventas), ventas.shape)

# Lista de columnas → DataFrame
subset = df[['Customer Name', 'Region', 'Sales']]
print(type(subset), subset.shape)

# Seleccionar por tipo de dato
numericas = df.select_dtypes(include='number')
textos    = df.select_dtypes(include='object')
print('Numéricas:', numericas.columns.tolist())
print('Texto:    ', textos.columns.tolist())


<class 'pandas.Series'> (9800,)
<class 'pandas.DataFrame'> (9800, 3)
Numéricas: ['Row ID', 'Postal Code', 'Sales']
Texto:     ['Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name']


C:\Users\alefe\AppData\Local\Temp\ipykernel_22840\4251535783.py:11: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  textos    = df.select_dtypes(include='object')


## Filtrado booleano

Una máscara booleana es una Serie de True/False con el mismo índice que el DataFrame. Al aplicarla, conserva solo las filas donde la máscara es True.

In [3]:
# Condición simple
mask_west = df['Region'] == 'West'
print(type(mask_west))
print(mask_west.value_counts())
print()

west = df[mask_west]
print(f'Filas de West: {len(west)}')

# Operadores: == != > < >= <=
grandes = df[df['Sales'] > 1000]
print(f'Ventas > 1000: {len(grandes)}')


<class 'pandas.Series'>
Region
False    6660
True     3140
Name: count, dtype: int64

Filas de West: 3140
Ventas > 1000: 462


In [4]:
# Múltiples condiciones — & (and), | (or), ~ (not)
# Los paréntesis son obligatorios alrededor de cada condición
west_tech = df[(df['Region'] == 'West') & (df['Category'] == 'Technology')]
print(f'West + Technology: {len(west_tech)}')

# Más de dos condiciones
seleccion = df[
    (df['Sales'] > 500) &
    (df['Category'] == 'Technology') &
    ~(df['Region'] == 'South')   # NOT South
]
print(f'Sales>500, Tech, no South: {len(seleccion)}')


West + Technology: 589
Sales>500, Tech, no South: 327


## isin() y between()

In [5]:
# isin — equivale a múltiples == unidos con |
regiones_objetivo = ['West', 'East']
df_costa = df[df['Region'].isin(regiones_objetivo)]
print(f'West o East: {len(df_costa)}')

# ~isin() — excluir una lista de valores
sin_office = df[~df['Category'].isin(['Office Supplies'])]
print(f'Sin Office Supplies: {len(sin_office)}')

# between — rango inclusivo en ambos extremos
rango_medio = df[df['Sales'].between(100, 500)]
print(f'Sales entre 100 y 500: {len(rango_medio)}')


West o East: 5925
Sin Office Supplies: 3891
Sales entre 100 y 500: 2553


## query()

`query()` acepta condiciones como string — más legible para condiciones complejas y permite usar variables externas con `@`.

In [6]:
# Equivalente a df[(df['Region'] == 'West') & (df['Sales'] > 500)]
resultado = df.query("Region == 'West' and Sales > 500")
print(len(resultado))

# Columnas con espacios van entre backticks
resultado2 = df.query("`Customer Name` == 'Sean Miller'")
print(resultado2[['Customer Name', 'Sales']])

# Variable externa con @
umbral = 1000
grandes = df.query('Sales > @umbral')
print(f'Sales > {umbral}: {len(grandes)}')


368
     Customer Name      Sales
2266   Sean Miller      7.968
2573   Sean Miller    526.450
2696   Sean Miller    821.300
2697   Sean Miller  22638.480
2698   Sean Miller     21.376
2699   Sean Miller      8.016
2700   Sean Miller     30.768
2701   Sean Miller     18.936
2702   Sean Miller    122.352
7853   Sean Miller      9.960
9187   Sean Miller      3.488
9188   Sean Miller     21.728
9189   Sean Miller    663.072
9190   Sean Miller     99.588
9191   Sean Miller     49.568
Sales > 1000: 462


## loc con condiciones

`loc` permite filtrar filas y seleccionar columnas en una sola expresión.

In [7]:
# loc[condición_filas, columnas]
resultado = df.loc[
    df['Category'] == 'Technology',
    ['Customer Name', 'Sub-Category', 'Sales']
]
print(resultado.head())
print()

# Útil para modificar valores en un subset sin SettingWithCopyWarning
df2 = df.copy()
df2.loc[df2['Sales'] < 0, 'Sales'] = 0   # reemplazar ventas negativas
print(f'Ventas negativas corregidas: {(df2["Sales"] < 0).sum()}')


         Customer Name Sub-Category     Sales
7      Brosina Hoffman       Phones   907.152
11     Brosina Hoffman       Phones   911.424
19  Zuschuss Donatelli       Phones   213.480
26       Eric Hoffmann  Accessories    90.570
35           Gene Hale       Phones  1097.544

Ventas negativas corregidas: 0


---
## Resumen

| Patrón | Sintaxis |
|--------|----------|
| Una columna | `df['col']` |
| Varias columnas | `df[['a', 'b']]` |
| Filtro simple | `df[df['col'] == valor]` |
| AND | `df[(cond1) & (cond2)]` |
| OR | `df[(cond1) \| (cond2)]` |
| NOT | `df[~condicion]` |
| Lista de valores | `df[df['col'].isin([...])]` |
| Rango | `df[df['col'].between(a, b)]` |
| String query | `df.query("col > 100 and otro == 'x'")` |
| Filtro + columnas | `df.loc[condicion, ['a', 'b']]` |
